# APIs: requêtes HTTP

## Extraction de texte et vérification du vocabulaire du corpus CAMille

J'ai commencé par télécharger les 51 PDF avec le S3_Scrape. 

Pour celui-ci je vais extraire le texte  dans le pdf et le mettre dans un tableau (DataFrame) avec le nom de chaque fichier

Pour API, Je vais utiliser l’API MediaWiki du Wiktionnaire français (https://fr.wiktionary.org) qui est accessible sans clé
On vérifie si les mots les plus fréquents des journaux existent dans le Wiktionnaire
Un mot absent peut indiquer une erreur de reconnaissance (OCR), un nom propre, ou une graphie ancienne

## Imports

In [8]:
import os
import glob
import pandas as pd
from pypdf import PdfReader  # bibliothèque pour lire les PDF

## Extraction du texte

In [ ]:
pdf_dir = "../../data/camille" 

# liste  qu'on remplira fichier par fichier
lignes = []

for chemin in sorted(glob.glob(os.path.join(pdf_dir, "*.pdf"))):
    lecteur = PdfReader(chemin)

    # chaque PDF ne contient qu'une page, on prend directement la page 0
    texte = lecteur.pages[0].extract_text() or ""

    lignes.append({
        "fichier": os.path.basename(chemin),  
        "texte": texte,
    })

Transformation de la liste en tableau

In [ ]:
df = pd.DataFrame(lignes)
print(df.shape)  # nombre de documents et de colonnes
df[["fichier"]].head()

(51, 2)


,fichier
0,KB_JB230_1892-08-07_01-0003.pdf
1,KB_JB230_1903-10-16_01-0002.pdf
2,KB_JB230_1913-07-05_01-0001.pdf
3,KB_JB258_1884-09-03_01-0003.pdf
4,KB_JB258_1894-12-09_01-0003.pdf


Affiche le début du texte du premier document

In [ ]:
print(df.loc[10, "fichier"])
print(df.loc[10, "texte"][:500])  # les 500 premiers caractères

KB_JB427_1933-01-04_01-00003.pdf
4 Ianvier 193S L/V LIBRE BELGIQUE 
EXTERIEUR 
*»• ' •* 
ALLEMAGNE 
LES AGRICULTEURS 
DE LA PRUSSE ORIENTALE 
DEMANDENT PROTECTION 
Kooiiicsbcrg. 3. — .-a chambre *!'*• 
priculture do la l'rus.s. Orientale u adres­
sé au chancelier von Schleicher et au 
miuietro do l'Intérieur vou Braun, un 
télé-zrammo demandant oue des mesures 
soient prises immédiatement nour empê­
cher rentrée en AHemanne des beurres, 
graisses, la-rd-s et orotluits analogues su­
perflus 
UNE MOTION HITLERIENNE 
CONTRE LES JO


## Compter les mots  et afficher les plus fréquents

In [17]:
import re
from collections import Counter  # compte automatiquement les occurrences

# on prend tout le texte du corpus, en minuscules
tout_le_texte = " ".join(df["texte"]).lower()

# on découpe en mots et on reccuperer le mots de 3 caractères ou plus
# les lettres accentuées sont incluses
mots = re.findall(r"[a-zàâäçéèêëîïôöùûüÿœæ]{3,}", tout_le_texte)

# on compte chaque mot et on garde les 20 plus fréquents
top20 = Counter(mots).most_common(20)
print(top20)

[('les', 4117), ('des', 2968), ('que', 2227), ('est', 2032), ('une', 1996), ('qui', 1794), ('pour', 1630), ('dans', 1616), ('par', 1580), ('sur', 1033), ('pas', 1007), ('plus', 828), ('aux', 742), ('nous', 683), ('ont', 665), ('avec', 654), ('son', 648), ('été', 640), ('sont', 586), ('elle', 565)]


## Pour en savoir plus

- Les API Rest: https://towardsdatascience.com/introduction-to-rest-apis-90b5d9676004
- La librairie [`requests`](https://requests.readthedocs.io/en/latest/)
- Développer une API en Python : https://openclassrooms.com/en/courses/4525361-realisez-un-dashboard-avec-tableau/5774811-creez-une-api-avec-flask